# 🔬 Notebook 3: Shopping Cart — Deep Dives (runnable)

## 🛠️ Setup

```bash
cd 06-system-designs/shopping-cart
uv sync
```

Select the `.venv` kernel in VS Code (top-right corner of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab only needs `pydantic` — no Redis, no database. We simulate everything in plain Python so you can focus on the *ideas*.


## What we'll build

Four mini-systems you can run and poke at:

1. **Inventory reservation with TTL** — why we don't decrement stock on *Add to Cart*.
2. **Checkout saga** — how to recover when payment fails after stock was reserved.
3. **Data hydration + short-TTL product cache** — how to show fresh prices without hammering the Product Service.
4. **Guest → user cart merge** — the idempotent algorithm.

We stay in pure Python — no Redis, no DB, no network. Every cell is standalone.

## Deep dive 1 — Inventory reservations with TTL

### The two naive choices (both bad)

| Strategy | Problem |
|---|---|
| Decrement stock on *Add to Cart* | Idle carts lock real buyers out for days |
| Decrement stock on *Pay* | Two people both pass the "in stock" check, one gets over-sold |

### The fix — reservation with TTL

At **checkout start** (not add-to-cart), create a **reservation** that decrements stock for a short window (e.g. 10 minutes). If payment succeeds, convert reservation → permanent sale. If TTL expires, the reservation **auto-releases** and stock goes back.

The class below simulates the inventory service. Notice the explicit `release()` method — we don't reach into its internals from outside.

In [ ]:
import time, threading, uuid

class Inventory:
    """In-memory inventory with time-boxed reservations."""
    def __init__(self, stock: dict[str, int]):
        self.stock = dict(stock)              # sku -> available units
        self.reservations: dict[str, tuple[str, int, float]] = {}
        self.lock = threading.Lock()          # single-machine concurrency

    def reserve(self, rid: str, sku: str, qty: int, ttl: float = 600) -> bool:
        with self.lock:
            self._sweep()
            if self.stock.get(sku, 0) < qty:
                return False
            self.stock[sku] -= qty
            self.reservations[rid] = (sku, qty, time.time() + ttl)
            return True

    def confirm(self, rid: str) -> bool:
        """Payment succeeded — reservation becomes a permanent sale."""
        with self.lock:
            return self.reservations.pop(rid, None) is not None

    def release(self, rid: str) -> bool:
        """Explicit rollback (payment failed / user cancelled)."""
        with self.lock:
            entry = self.reservations.pop(rid, None)
            if entry is None:
                return False
            sku, qty, _ = entry
            self.stock[sku] += qty
            return True

    def _sweep(self) -> None:
        """Reclaim stock from expired reservations."""
        now = time.time()
        for rid in [k for k, (_, _, exp) in self.reservations.items() if exp < now]:
            sku, qty, _ = self.reservations.pop(rid)
            self.stock[sku] += qty

# Scenario: 2 units in stock, Alice reserves 1 with a 1 s TTL and ghosts.
inv = Inventory({"BOOK": 2})
print("alice reserves 1:", inv.reserve("res-alice", "BOOK", 1, ttl=1))
print("bob tries 2     :", inv.reserve("res-bob1",  "BOOK", 2))   # only 1 left
print("bob takes 1     :", inv.reserve("res-bob2",  "BOOK", 1))
print("stock now       :", inv.stock)
print()
print("... 1.1 s passes, alice's TTL expires ...")
time.sleep(1.1)
inv._sweep()
print("stock now       :", inv.stock)   # alice's unit came back

## Deep dive 2 — Checkout saga with clean compensation

Checkout spans **Inventory → Payment → Order**. You can't wrap them in one DB transaction (they're separate services). Instead: run them in sequence, and if one fails, **compensate** the previous step.

```
┌─ reserve stock ─► charge card ─► create order ─┐
│       │                │                       │
│       │(fail)          │(fail)                 ▼
▼       ▼                ▼                    success
abort  ✓           release() stock
```

Below we use the **clean `release()` API** instead of poking internals. This makes the saga readable and testable.

In [ ]:
class FakeCard:
    """Payment gateway stub."""
    def __init__(self, ok: bool = True): self.ok = ok
    def charge(self, amount: int) -> dict:
        if not self.ok:
            raise RuntimeError("card declined")
        return {"txn": f"tx_{amount}"}

def checkout(inv: Inventory, card: FakeCard,
             cart_id: str, sku: str, qty: int, amount: int) -> dict:
    rid = f"res-{cart_id}"

    # Step 1: reserve stock
    if not inv.reserve(rid, sku, qty, ttl=60):
        return {"status": "out_of_stock"}

    # Step 2: charge card (with compensation on failure)
    try:
        txn = card.charge(amount)
    except Exception as e:
        inv.release(rid)                           # ← compensate
        return {"status": "payment_failed", "error": str(e)}

    # Step 3: confirm the reservation (permanent decrement)
    inv.confirm(rid)
    # (Real systems would also write an Order record here.)
    return {"status": "paid", "txn": txn, "cart_id": cart_id}

inv = Inventory({"BOOK": 5})
print(checkout(inv, FakeCard(ok=True),  "c1", "BOOK", 2, 20))
print(checkout(inv, FakeCard(ok=False), "c2", "BOOK", 1, 10))   # fails, releases
print(checkout(inv, FakeCard(ok=True),  "c3", "BOOK", 1, 10))
print("final stock:", inv.stock)   # 5 - 2 - 1 = 2

### Why not 2PC (two-phase commit)?

2PC *can* coordinate a transaction across services, but it blocks resources while voting and falls apart when any coordinator dies. Sagas trade strict atomicity for **availability and partial-failure recovery**, which is exactly what e-commerce wants.

## Deep dive 3 — Data hydration + short-TTL product cache

Remember from NB1: the Cart DB stores `{sku, qty}` only. At **view** time, the Cart Service must fetch the current price/name from the **Product Service**. That's the **hydration** step.

### Why you need a cache in front

If every cart view calls the Product Service for every item, and carts are viewed 150 000× / second at peak, the Product Service melts. Put a **5-minute TTL** in front of it. Prices only need to be accurate *to the minute*; the hard price check happens at checkout anyway.

In [ ]:
import time
from decimal import Decimal

# --- Product Service stub -------------------------------------------------
class ProductService:
    def __init__(self, catalog: dict[str, dict]):
        self.catalog = catalog
        self.calls = 0                           # count network hits

    def get(self, sku: str) -> dict:
        self.calls += 1                          # simulate a network call
        return self.catalog[sku].copy()

# --- Short-TTL cache ------------------------------------------------------
class ProductCache:
    def __init__(self, svc: ProductService, ttl: float = 300):
        self.svc = svc
        self.ttl = ttl
        self.cache: dict[str, tuple[dict, float]] = {}

    def get(self, sku: str) -> dict:
        now = time.time()
        entry = self.cache.get(sku)
        if entry and now - entry[1] < self.ttl:
            return entry[0]                      # cache hit
        fresh = self.svc.get(sku)                # miss → fetch + store
        self.cache[sku] = (fresh, now)
        return fresh

# --- Cart (slim) + hydration ---------------------------------------------
def view_cart(cart_items: list[dict], cache: ProductCache) -> dict:
    hydrated, total = [], Decimal(0)
    for it in cart_items:
        p = cache.get(it["sku"])
        line_total = Decimal(str(p["price"])) * it["qty"]
        hydrated.append({**it, "name": p["name"], "price": p["price"],
                         "line_total": str(line_total)})
        total += line_total
    return {"items": hydrated, "total": str(total)}

svc = ProductService({
    "A": {"name": "Python Crash Course", "price": "29.99"},
    "B": {"name": "Clean Code",          "price": "35.00"},
})
cache = ProductCache(svc, ttl=300)

slim_cart = [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]

print("first view  :", view_cart(slim_cart, cache))
print("product-svc calls so far:", svc.calls)   # 2

print("second view :", view_cart(slim_cart, cache))
print("product-svc calls so far:", svc.calls)   # still 2 — cache hit

### Graceful degradation

What if the Product Service is **down** when a user opens their cart?

- **Bad:** 500 error, user sees nothing.
- **Best:** show cached data if we have it (even if stale); if no cache, show items with `"Loading price..."` placeholders and a banner. The cart itself (sku + qty) still works because it's in Redis.

This is called **graceful degradation** — degrade the feature, don't take the page down. Try implementing the stale-on-error path as an exercise.

## Deep dive 4 — Guest → User cart merge

Scenario: a shopper browses anonymously (`guest_session_id = "gs_abc"`), adds items, then logs in.

The reference spec:
- **Overlap** (same sku in both carts) → **sum quantities**, capped at the per-item limit.
- **No overlap** → add guest's items to the user cart.
- **Delete the guest cart** after merge so a retry doesn't double up.
- The whole thing must be **idempotent** — the client may retry on a flaky connection.

In [ ]:
from dataclasses import dataclass, field

MAX_QTY_PER_ITEM = 10

@dataclass
class CartStore:
    """Toy cart DB. Keyed by user_id *or* guest_session_id — both strings."""
    carts: dict[str, dict[str, int]] = field(default_factory=dict)  # owner -> {sku:qty}
    merges: dict[str, dict] = field(default_factory=dict)           # merge_id -> result

    def get(self, owner: str) -> dict[str, int]:
        return self.carts.get(owner, {}).copy()

    def put(self, owner: str, items: dict[str, int]) -> None:
        self.carts[owner] = items

    def delete(self, owner: str) -> None:
        self.carts.pop(owner, None)

    def merge(self, guest_id: str, user_id: str, merge_id: str) -> dict[str, int]:
        # 🔑 Idempotency: same merge_id returns the same answer.
        if merge_id in self.merges:
            return self.merges[merge_id]

        guest = self.get(guest_id)
        user  = self.get(user_id)

        merged = dict(user)
        for sku, qty in guest.items():
            merged[sku] = min(merged.get(sku, 0) + qty, MAX_QTY_PER_ITEM)

        self.put(user_id, merged)
        self.delete(guest_id)
        self.merges[merge_id] = merged
        return merged

# Scenario: guest has 2×A, 1×B. User (from last week) has 1×A, 1×C.
db = CartStore()
db.put("gs_abc", {"A": 2, "B": 1})
db.put("u_42",   {"A": 1, "C": 1})

print("1st merge :", db.merge("gs_abc", "u_42", "m-001"))
# {'A': 3, 'C': 1, 'B': 1}   (A summed 2+1, cap not hit)

print("retry     :", db.merge("gs_abc", "u_42", "m-001"))
# Identical output — guest cart is already gone but the merge_id replays.

# Hoarding attempt: guest with 50 copies of A
db.put("gs_hoard", {"A": 50})
db.put("u_99", {"A": 2})
print("cap       :", db.merge("gs_hoard", "u_99", "m-002"))
# {'A': 10}  — capped at MAX_QTY_PER_ITEM

## Closing thoughts

| Pattern | Where |
|---|---|
| **Slim source-of-truth + live hydration** | Any service where a field can change (prices, permissions, user names) |
| **Reservation with TTL** | Any shared scarce resource (event seats, ride drivers, ad impressions) |
| **Saga with compensation** | Any cross-service workflow that can't share a DB transaction |
| **Idempotency keys** | Any retryable action that moves money or mutates state |
| **Short-TTL metadata cache** | Any fan-out that amplifies backend load |
| **Per-item qty caps** | Any user-controlled collection (carts, follows, uploads) |

### Exercises

1. Make the merge run in **parallel reads** (one fetch for guest, one for user) using `threading` or `asyncio`.
2. Add a **stale-on-error** fallback to `ProductCache` — if the upstream call fails, serve the last cached value with a `stale=True` flag.
3. Extend `Inventory` with a **Redis-like script** so `reserve` is atomic across processes (hint: Lua script or `WATCH/MULTI/EXEC`).
4. Write a test that simulates 1000 concurrent checkouts of the last unit in stock — only one should win.